# 🔭 EDA of MaNGA integrated spectra + morphology metadata
### ~16k `.fits` spectra  ×  `morfologia.csv` (16k galaxies) — with a focus on **Early- vs Late-Type galaxies**

**Inputs (structure taken from the sample files):**

| Source | Key | Content |
|---|---|---|
| `morfologia.csv` | `name` (e.g. `manga-10001-12701`) | `objra, objdec, Type, Ttype, nsa_z, LogMass, incl, PETRO_TH90` |
| `manga-PLATE-IFU.fits` (one per galaxy) | file name = `name` | `WAVE, FLUX_1RE, FLUX_FoV, ERROR_1RE, ERROR_FoV` |

**What you'll see in the sample data:**
* `WAVE` is in the **observed frame** (≈3622–10351 Å, 1.5 Å step). Masked pixels are **removed**, which leaves gaps, and the red end is padded with `FLUX = 0`.
* `FLUX_1RE` is the spectrum inside 1 effective radius, and `FLUX_FoV` is the full IFU field of view.
* Metadata uses `nsa_z = -9999` and empty cells for missing values.

**Notebook roadmap**
0. Setup & configuration  
1. Metadata EDA  
2. Spectra file inventory & matching (by galaxy name)  
3. Reading one spectrum & a deep dive into one galaxy  
4. Bulk processing of all spectra (parallel, cached): QC stats + rest-frame resampling  
5. Spectral data-quality EDA  
6. Global spectral EDA (stacks, PCA)  
7. Spectral feature measurement (D4000, Lick indices, emission-line EWs)  
8. **How the spectra & metadata relate to Early-Type (ETG) vs Late-Type (LTG) classification**  
9. Export  

> **Colab tips:** Reading 16k small files straight from Google Drive is slow. Put the spectra in a single `spectra.zip` on Drive. Section 0 unzips it to the local `/content` disk. Bulk results are cached to Drive, so you only pay the cost once.

## 0. Setup & configuration

In [ ]:
import os, sys, re, time, warnings, subprocess, zipfile
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
from scipy import stats
from tqdm.auto import tqdm
from IPython.display import display, Markdown

IN_COLAB = 'google.colab' in sys.modules

# astropy is pre-installed on Colab; install it only if missing
try:
    from astropy.io import fits
except ImportError:
    try:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'astropy'], check=True)
        from astropy.io import fits
    except Exception as e:
        fits = None
        print('⚠️ astropy not available – only .csv spectra can be read:', e)

warnings.filterwarnings('ignore', category=RuntimeWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams.update({'figure.dpi': 110, 'axes.titleweight': 'bold', 'figure.max_open_warning': 0})
pd.set_option('display.max_columns', 80); pd.set_option('display.width', 220)
print('Colab:', IN_COLAB, '| numpy', np.__version__, '| pandas', pd.__version__)

In [ ]:
# ======================= CONFIGURATION (edit here) =======================
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

DATA_DIR   = Path(os.environ.get('MANGA_DATA_DIR', '/content/drive/MyDrive/manga_eda'))
META_PATH  = DATA_DIR / 'morfologia.csv'          # metadata (16k rows)
SPEC_DIR   = DATA_DIR / 'spectra'                 # folder with manga-*.fits (sub-folders OK)
SPEC_ZIP   = DATA_DIR / 'spectra.zip'             # OPTIONAL: zip with all spectra (much faster on Colab)
LOCAL_DIR  = Path('/content/spectra_local')       # where the zip is extracted on Colab
CACHE_DIR  = DATA_DIR / 'eda_cache'               # cached arrays / tables / figures

SPEC_PATTERNS = ('*.fits', '*.fits.gz', '*.fit', '*.csv')   # .csv twins of the FITS also work
NAME_REGEX    = re.compile(r'(manga-\d+-\d+)', re.I)        # how the galaxy name is found in a file name
MAX_FILES     = None      # e.g. 500 for a quick dry run; None = all
N_THREADS     = 32        # I/O threads for reading files
FORCE_RELOAD  = False     # True = ignore cache and re-read all spectra

# Column names inside the spectra
WAVE_COLS  = ('WAVE', 'WAVELENGTH', 'LAMBDA', 'LOGLAM')
APERTURES  = {'1RE': ('FLUX_1RE', 'ERROR_1RE'),   # inside 1 effective radius
              'FoV': ('FLUX_FoV', 'ERROR_FoV')}   # full IFU field of view
WAVE_IN_VACUUM = True     # MaNGA DRP wavelengths are vacuum; index band definitions (air) get converted

# Grids
OBS_STEP  = 1.5
OBS_GRID  = np.arange(3622.0, 10351.0 + OBS_STEP / 2, OBS_STEP)   # observed-frame grid of the sample files
REST_GRID = np.arange(3550.0, 9200.0, 2.0)                        # common rest-frame grid
NORM_WINDOW = (5300.0, 5700.0)                                    # rest-frame normalisation window
SN_MIN = 3.0              # minimum S/N (per pixel, in NORM_WINDOW) for a spectrum to be "good"

# Morphological classes from Ttype (-5=E, -2=S0, 0=S0a, 1=Sa ... 7=Sd ...)
ETG_MAX_TTYPE   = 0       # ETG: Ttype <= 0 (E, S0, S0a). Use -1 to treat S0a as LTG/transition
LTG_TTYPE_RANGE = (1, 10) # LTG: Sa ... Irr. Codes outside both (e.g. 11 = 'S' in the sample) -> 'Other'

SEED = 42
CLASS_COLORS = {'ETG': '#C0392B', 'LTG': '#2E86C1', 'Other': '#95A5A6', 'Unknown': '#D5D8DC'}
TTYPE_CMAP = mpl.colormaps['RdYlBu']     # red = early, blue = late
# ========================================================================

CACHE_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR = CACHE_DIR / 'figures'; FIG_DIR.mkdir(exist_ok=True)
rng = np.random.default_rng(SEED)

def savefig(name):
    plt.savefig(FIG_DIR / f'{name}.png', bbox_inches='tight', dpi=130)

print('Metadata :', META_PATH, '| exists:', META_PATH.exists())
print('Spectra  :', SPEC_DIR, '| exists:', SPEC_DIR.exists())
print('Zip      :', SPEC_ZIP, '| exists:', SPEC_ZIP.exists())

In [ ]:
# Optional: extract spectra.zip to the fast local disk (only once per Colab session)
if SPEC_ZIP.exists():
    if not LOCAL_DIR.exists() or not any(LOCAL_DIR.rglob('*')):
        t0 = time.time(); LOCAL_DIR.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(SPEC_ZIP) as zf:
            zf.extractall(LOCAL_DIR)
        print(f'Extracted {SPEC_ZIP.name} in {time.time()-t0:.0f}s')
    SPEC_DIR = LOCAL_DIR
print('Reading spectra from:', SPEC_DIR)

## 1. Metadata EDA (`morfologia.csv`)
### 1.1 Load & clean
* Sentinel values (`-9999`) and blanks → `NaN`.
* Parse `plate` and `ifudesign` from the name. The IFU size (19/37/61/91/127 fibres) is `ifudesign // 100`.
* Morphological class from `Ttype`: **ETG** (E, S0, S0a), **LTG** (Sa → Irr), and **Other** (codes outside the configured ranges).
* The bar flag comes from the Hubble `Type` string: `SB` = strong bar, `SAB` = weak bar.
* Physical size R90 [kpc] = `PETRO_TH90` (arcsec) × angular-diameter distance (flat ΛCDM, H0=70, Ωm=0.3).

In [ ]:
meta_raw = pd.read_csv(META_PATH)
print('Raw shape:', meta_raw.shape)
display(meta_raw.head())
meta_raw.info()

In [ ]:
SENTINELS = [-9999, -9999.0, -999, -99]
NUM_COLS = ['objra', 'objdec', 'Ttype', 'nsa_z', 'LogMass', 'incl', 'PETRO_TH90']

meta = meta_raw.copy()
meta.columns = [c.strip() for c in meta.columns]
meta['name'] = meta['name'].astype(str).str.strip().str.lower()
meta['Type'] = meta['Type'].astype(str).str.strip().replace({'nan': np.nan, '': np.nan})
for c in NUM_COLS:
    meta[c] = pd.to_numeric(meta[c], errors='coerce')
    meta.loc[meta[c].isin(SENTINELS), c] = np.nan
meta.loc[meta['nsa_z'] <= 0, 'nsa_z'] = np.nan

# plate / IFU
pi = meta['name'].str.extract(r'manga-(\d+)-(\d+)')
meta['plate'] = pd.to_numeric(pi[0], errors='coerce').astype('Int64')
meta['ifudesign'] = pd.to_numeric(pi[1], errors='coerce').astype('Int64')
meta['ifu_size'] = (meta['ifudesign'] // 100).astype('Int64')

# morphology classes
def morph_class(t):
    if pd.isna(t): return 'Unknown'
    if t <= ETG_MAX_TTYPE: return 'ETG'
    if LTG_TTYPE_RANGE[0] <= t <= LTG_TTYPE_RANGE[1]: return 'LTG'
    return 'Other'

def morph_group(t):
    if pd.isna(t): return 'Unknown'
    if t <= -3: return '1 E'
    if t <= 0:  return '2 S0/S0a'
    if t <= 3:  return '3 Sa–Sb'
    if t <= 7:  return '4 Sbc–Sd'
    if t <= 10: return '5 Sdm–Irr'
    return '6 Other'

meta['morph_class'] = meta['Ttype'].map(morph_class)
meta['morph_group'] = meta['Ttype'].map(morph_group)

def bar_flag(t):
    if not isinstance(t, str) or not t.startswith('S'): return 'n/a'
    if t.startswith('SAB'): return 'weak bar (SAB)'
    if t.startswith('SB'):  return 'strong bar (SB)'
    return 'no bar'
meta['bar'] = meta['Type'].map(bar_flag)

# physical size
def ang_diam_dist_mpc(z, H0=70.0, Om=0.3):
    z = np.asarray(z, float); out = np.full(z.shape, np.nan)
    ok = np.isfinite(z) & (z > 0)
    if ok.any():
        zz = np.linspace(0, z[ok].max() * 1.01, 5000)
        iEz = 1 / np.sqrt(Om * (1 + zz) ** 3 + 1 - Om)
        dc = 299792.458 / H0 * np.concatenate([[0], np.cumsum(0.5 * (iEz[1:] + iEz[:-1]) * np.diff(zz))])
        out[ok] = np.interp(z[ok], zz, dc) / (1 + z[ok])
    return out
meta['DA_Mpc'] = ang_diam_dist_mpc(meta['nsa_z'])
meta['R90_kpc'] = meta['PETRO_TH90'] * meta['DA_Mpc'] * 1e3 * np.pi / (180 * 3600)

# duplicates
dups = meta['name'].duplicated(keep='first')
print('Duplicated names:', int(dups.sum()))
meta = meta[~dups].reset_index(drop=True)
print('Clean shape:', meta.shape)
display(meta.head())

### 1.2 Missing values & summary statistics

In [ ]:
miss = meta_raw.replace(SENTINELS, np.nan).isna().mean().mul(100).rename('% missing (raw, sentinels→NaN)')
display(miss.to_frame().T.round(2))

fig, ax = plt.subplots(figsize=(9, 3.2))
sample = meta[NUM_COLS + ['Type']].isna()
sns.heatmap(sample.sample(min(2000, len(sample)), random_state=SEED).sort_index().T,
            cbar=False, cmap=['#EAF2F8', '#C0392B'], ax=ax, xticklabels=False)
ax.set_title('Missing-value map (red = missing, random 2000 rows)'); plt.tight_layout(); savefig('meta_missing'); plt.show()

display(meta[['nsa_z', 'LogMass', 'incl', 'PETRO_TH90', 'R90_kpc', 'Ttype']].describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]).T.round(3))

### 1.3 Distributions of physical properties

In [ ]:
cols = ['nsa_z', 'LogMass', 'incl', 'PETRO_TH90', 'R90_kpc', 'Ttype']
labels = {'nsa_z': 'redshift z (NSA)', 'LogMass': 'log M* [M☉]', 'incl': 'inclination [deg]',
          'PETRO_TH90': 'Petrosian R90 [arcsec]', 'R90_kpc': 'R90 [kpc]', 'Ttype': 'T-type'}
fig, axes = plt.subplots(2, 3, figsize=(15, 7.5))
for ax, c in zip(axes.ravel(), cols):
    x = meta[c].dropna()
    if c == 'Ttype':
        vc = x.value_counts().sort_index()
        ax.bar(vc.index, vc.values, color=[TTYPE_CMAP((t + 5) / 16) for t in vc.index], edgecolor='k')
    else:
        lo, hi = x.quantile([.005, .995])
        ax.hist(x.clip(lo, hi), bins=60, color='#5D6D7E')
        ax.axvline(x.median(), color='crimson', ls='--', label=f'median={x.median():.3g}'); ax.legend()
    ax.set_title(labels[c]); ax.set_ylabel('N')
plt.tight_layout(); savefig('meta_distributions'); plt.show()

### 1.4 Morphological types

In [ ]:
order = (meta.groupby('Type')['Ttype'].median().sort_values().index)
fig, axes = plt.subplots(1, 2, figsize=(16, 4.8), gridspec_kw={'width_ratios': [3, 1]})
vc = meta['Type'].value_counts().reindex(order)
axes[0].bar(range(len(vc)), vc.values,
            color=[TTYPE_CMAP((meta.loc[meta.Type == t, 'Ttype'].median() + 5) / 16) for t in vc.index], edgecolor='k')
axes[0].set_xticks(range(len(vc))); axes[0].set_xticklabels(vc.index, rotation=90)
axes[0].set_title('Hubble Type counts (ordered by T-type)'); axes[0].set_ylabel('N')
cc = meta['morph_class'].value_counts()
axes[1].pie(cc.values, labels=[f'{k}\n{v} ({v/cc.sum():.0%})' for k, v in cc.items()],
            colors=[CLASS_COLORS[k] for k in cc.index], startangle=90, wedgeprops={'edgecolor': 'w'})
axes[1].set_title('Morphological class')
plt.tight_layout(); savefig('meta_types'); plt.show()

# Consistency check Type <-> Ttype: every Type should map to a single Ttype
chk = meta.groupby('Type')['Ttype'].agg(['nunique', 'min', 'max', 'count']).sort_values('min')
print('Types mapping to >1 Ttype value:'); display(chk[chk['nunique'] > 1])
print("'Other' / 'Unknown' class content:"); display(meta.loc[meta.morph_class.isin(['Other', 'Unknown']), ['Type', 'Ttype']].value_counts(dropna=False).to_frame('N'))

### 1.5 Sky coverage, selection & correlations

In [ ]:
fig = plt.figure(figsize=(16, 5))
ax = fig.add_subplot(121, projection='mollweide')
ra = np.radians(meta['objra'].values); ra = np.where(ra > np.pi, ra - 2 * np.pi, ra)
for k, g in meta.groupby('morph_class'):
    r = np.radians(g['objra'].values); r = np.where(r > np.pi, r - 2 * np.pi, r)
    ax.scatter(-r, np.radians(g['objdec']), s=2, alpha=.5, color=CLASS_COLORS[k], label=k)
ax.set_xticklabels([]); ax.set_title('Sky distribution (Mollweide)'); ax.legend(markerscale=5, loc='lower right')

ax = fig.add_subplot(122)
sc = ax.scatter(meta['nsa_z'], meta['LogMass'], c=meta['Ttype'], cmap=TTYPE_CMAP, vmin=-5, vmax=10, s=4, alpha=.6)
plt.colorbar(sc, ax=ax, label='T-type'); ax.set_xscale('log')
ax.set_xlabel('z'); ax.set_ylabel('log M*'); ax.set_title('Redshift–mass plane (MaNGA selection)')
plt.tight_layout(); savefig('meta_sky_zmass'); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(15, 4.8))
sns.boxplot(data=meta, x='ifu_size', y='PETRO_TH90', ax=axes[0], color='#AED6F1', fliersize=1)
axes[0].set_title('Galaxy angular size vs IFU size (fibres)'); axes[0].set_xlabel('IFU size (fibres)')
corr = meta[['nsa_z', 'LogMass', 'incl', 'PETRO_TH90', 'R90_kpc', 'Ttype', 'ifu_size']].astype(float).corr(method='spearman')
sns.heatmap(corr, annot=True, fmt='.2f', cmap='vlag', center=0, ax=axes[1], square=True)
axes[1].set_title('Spearman correlation (metadata)')
plt.tight_layout(); savefig('meta_ifu_corr'); plt.show()

## 2. Spectra inventory & matching to metadata
Each file is matched to the metadata through the galaxy name found in the file name (`manga-PLATE-IFU`). If both `.fits` and `.csv` exist for the same galaxy, the FITS file is used.

In [ ]:
t0 = time.time()
paths = []
for pat in SPEC_PATTERNS:
    paths += list(Path(SPEC_DIR).rglob(pat))
inv = pd.DataFrame({'path': paths})
inv['fname'] = inv['path'].map(lambda p: p.name)
inv['ext'] = inv['fname'].str.extract(r'(\.fits\.gz|\.fits|\.fit|\.csv)$', flags=re.I)[0].str.lower()
inv['name'] = inv['fname'].map(lambda s: (m.group(1).lower() if (m := NAME_REGEX.search(s)) else None))
inv['size_kB'] = inv['path'].map(lambda p: p.stat().st_size / 1024)
prio = {'.fits': 0, '.fits.gz': 1, '.fit': 2, '.csv': 3}
inv = (inv.dropna(subset=['name']).assign(prio=inv['ext'].map(prio))
          .sort_values(['name', 'prio']).drop_duplicates('name').drop(columns='prio').reset_index(drop=True))
print(f'{len(inv)} spectra found in {time.time()-t0:.1f}s | extensions:', inv['ext'].value_counts().to_dict())

s_meta, s_files = set(meta['name']), set(inv['name'])
both, only_meta, only_files = s_meta & s_files, s_meta - s_files, s_files - s_meta
print(f'Matched: {len(both)} | in metadata without spectrum: {len(only_meta)} | spectra without metadata: {len(only_files)}')
print('examples metadata-only:', sorted(only_meta)[:5]); print('examples file-only:', sorted(only_files)[:5])

fig, axes = plt.subplots(1, 2, figsize=(13, 3.6))
axes[0].bar(['matched', 'meta only', 'file only'], [len(both), len(only_meta), len(only_files)],
            color=['#27AE60', '#F39C12', '#C0392B'])
axes[0].set_title('Name matching')
axes[1].hist(inv['size_kB'], bins=60, color='#5D6D7E'); axes[1].set_xlabel('file size [kB]'); axes[1].set_title('File sizes')
plt.tight_layout(); savefig('inventory'); plt.show()
meta['has_spec'] = meta['name'].isin(s_files)
display(pd.crosstab(meta['morph_class'], meta['has_spec'], margins=True))

In [ ]:
# Peek inside one FITS file: HDU structure, header, columns
fits_example = inv.loc[inv['ext'].isin(['.fits', '.fits.gz', '.fit']), 'path']
if len(fits_example) and fits is not None:
    with fits.open(fits_example.iloc[0]) as hdul:
        hdul.info()
        for h in hdul:
            if isinstance(h, (fits.BinTableHDU, fits.TableHDU)):
                print('\nColumns of', h.name, ':'); print(h.columns)
        print('\nPrimary header (first 40 cards):'); print(repr(hdul[0].header[:40]))
else:
    print('No FITS files found (or astropy missing) – using CSV spectra.')

## 3. Reading spectra
`read_spectrum()` returns a standard dictionary `{wave, flux_1RE, err_1RE, flux_FoV, err_FoV}` and handles these layouts:
* a FITS **binary table** with one row per pixel (the same layout as the CSV samples),
* a FITS binary table with **one row of array columns**,
* FITS **image extensions** named `WAVE`, `FLUX_1RE`, …,
* `.csv` twins.

A pixel counts as **valid** if its flux and error are finite, `error > 0`, and `flux ≠ 0` (the zero padding at the red end is treated as invalid).

In [ ]:
def _pick(names, target):
    up = {n.upper(): n for n in names}
    return up.get(target.upper())

def read_spectrum(path):
    path = Path(path); out = {}
    if path.suffix.lower() == '.csv':
        df = pd.read_csv(path); names = list(df.columns)
        get = lambda c: df[c].to_numpy(dtype=float)
    else:
        if fits is None: raise ImportError('astropy is required for FITS files')
        with fits.open(path, memmap=False) as hdul:
            table = None
            for h in hdul:
                if isinstance(h, (fits.BinTableHDU, fits.TableHDU)) and h.data is not None:
                    if any(_pick(h.columns.names, k) for k in WAVE_COLS):
                        table = h; break
            if table is not None:
                names = list(table.columns.names)
                cols = {n: np.asarray(table.data[n], dtype=float).ravel() for n in names}
            else:  # image-extension layout
                cols = {h.name: np.asarray(h.data, dtype=float).ravel() for h in hdul if h.data is not None}
                names = list(cols)
        get = lambda c: cols[c]
    wcol = next((_pick(names, k) for k in WAVE_COLS if _pick(names, k)), None)
    if wcol is None: raise KeyError(f'no wavelength column in {path.name}: {names}')
    wave = get(wcol)
    if wcol.upper() == 'LOGLAM': wave = 10 ** wave
    out['wave'] = wave
    for ap, (fc, ec) in APERTURES.items():
        f, e = _pick(names, fc), _pick(names, ec)
        out[f'flux_{ap}'] = get(f) if f else np.full(wave.shape, np.nan)
        out[f'err_{ap}']  = get(e) if e else np.full(wave.shape, np.nan)
    o = np.argsort(out['wave'])
    return {k: v[o] for k, v in out.items()}

def valid_mask(f, e):
    return np.isfinite(f) & np.isfinite(e) & (e > 0) & (f != 0)

def der_snr(f):
    # DER_SNR (Stoehr et al. 2008): S/N estimated from the flux alone, independent of the error column
    f = f[np.isfinite(f)]
    if len(f) < 10: return np.nan
    noise = 1.482602 / np.sqrt(6) * np.median(np.abs(2 * f[2:-2] - f[:-4] - f[4:]))
    return np.median(f) / noise if noise > 0 else np.nan

def to_grid(w, f, valid, grid, max_gap):
    # Linear resampling without bridging masked gaps wider than max_gap (those become NaN)
    wv, fv = w[valid], f[valid]
    out = np.full(grid.shape, np.nan)
    if len(wv) < 10: return out
    out = np.interp(grid, wv, fv, left=np.nan, right=np.nan)
    idx = np.clip(np.searchsorted(wv, grid), 1, len(wv) - 1)
    out[(wv[idx] - wv[idx - 1]) > max_gap] = np.nan
    return out

spec0 = read_spectrum(inv['path'].iloc[0])
print(inv['fname'].iloc[0], {k: v.shape for k, v in spec0.items()})

### 3.1 Single-galaxy deep dive
Shaded regions are wavelength ranges with no valid pixels. Dashed lines show the main spectral features redshifted with `nsa_z`.

In [ ]:
LINES = {'[OII]': 3727.4, 'CaII K': 3933.7, 'CaII H': 3968.5, 'Hδ': 4101.7, 'G': 4304.4, 'Hγ': 4340.5, 'Hβ': 4861.3,
         '[OIII]': 5006.8, 'Mgb': 5175.4, 'NaD': 5892.9, 'Hα': 6562.8, '[NII]': 6583.4, '[SII]': 6724.0}

def plot_galaxy(name, zoom=None):
    row = meta.set_index('name').loc[name]
    s = read_spectrum(inv.set_index('name').loc[name, 'path'])
    w, z = s['wave'], row['nsa_z']
    fig, axes = plt.subplots(3, 1, figsize=(15, 9), sharex=True, gridspec_kw={'height_ratios': [2, 2, 1]})
    for ax, ap in zip(axes[:2], APERTURES):
        f, e = s[f'flux_{ap}'], s[f'err_{ap}']; v = valid_mask(f, e)
        ax.plot(w[v], f[v], lw=.6, color='k')
        ax.fill_between(w[v], (f - e)[v], (f + e)[v], color='orange', alpha=.3, lw=0, label='±1σ (error column)')
        dw = np.diff(w[v])
        for i in np.where(dw > 2.5 * OBS_STEP)[0]:
            ax.axvspan(w[v][i], w[v][i + 1], color='grey', alpha=.15)
        lo, hi = np.nanpercentile(f[v], [0.5, 99.5]) if v.any() else (0, 1)
        ax.set_ylim(min(0, lo), hi * 1.3); ax.set_ylabel(f'FLUX_{ap}')
        ax.set_title(f'{ap}: valid pixels {v.sum()}/{len(w)}, median S/N(err)={np.nanmedian(f[v]/e[v]):.1f}, DER_SNR={der_snr(f[v]):.1f}', fontsize=10)
        if np.isfinite(z):
            for lab, lw0 in LINES.items():
                ax.axvline(lw0 * (1 + z), color='teal', ls='--', lw=.6, alpha=.6)
                ax.text(lw0 * (1 + z), hi * 1.25, lab, rotation=90, fontsize=7, va='top', color='teal')
        ax.legend(loc='upper right', fontsize=8)
    for ap, c in zip(APERTURES, ['k', 'purple']):
        f, e = s[f'flux_{ap}'], s[f'err_{ap}']; v = valid_mask(f, e)
        axes[2].plot(w[v], (f / e)[v], lw=.5, color=c, label=ap)
    axes[2].set_yscale('log'); axes[2].set_ylabel('S/N per pixel'); axes[2].set_xlabel('observed wavelength [Å]'); axes[2].legend()
    if zoom: axes[2].set_xlim(*zoom)
    fig.suptitle(f"{name} | {row['Type']} (T={row['Ttype']}) | {row['morph_class']} | z={z:.4f} | logM*={row['LogMass']:.2f}", fontweight='bold')
    plt.tight_layout(); plt.show()

example = meta.loc[meta['has_spec']].sample(1, random_state=SEED)['name'].iloc[0]
plot_galaxy(example)

## 4. Bulk processing of all spectra (parallel + cached)
For each galaxy we compute:
* **QC statistics:** number of pixels, wavelength range, gaps, fraction of zero/negative flux, median S/N from the error column, and **DER_SNR**, a noise estimate from the flux alone that lets us check whether the error columns are realistic.
* a **rest-frame** spectrum for each aperture (`λ/(1+z)`, `F·(1+z)`) on `REST_GRID`. Gaps are not interpolated across.
* a valid-pixel mask on the observed-frame grid, used to find where masked pixels concentrate (sky lines, detector edges).

Arrays are stored as `float32` (≈180 MB per array for 16k × 2825 pixels) and cached to `CACHE_DIR`.

In [ ]:
def process_one(name, path, z):
    rec = {'name': name, 'ok': False}
    try:
        s = read_spectrum(path)
    except Exception as ex:
        rec['error'] = repr(ex)[:200]
        return rec, None, None
    w = s['wave']
    dw = np.diff(w); step = float(np.median(dw)) if len(dw) else np.nan
    rec.update(n_pix=len(w), wave_min=float(w.min()), wave_max=float(w.max()), step_med=step,
               n_gaps=int((dw > 2.5 * step).sum()), max_gap_A=float(dw.max()) if len(dw) else np.nan)
    rec['frac_removed_pix'] = 1 - len(w) / (round((w.max() - w.min()) / step) + 1)
    zf = z if (z is not None and np.isfinite(z) and z > 0) else np.nan
    rest, obs_valid = {}, np.zeros(len(OBS_GRID), np.uint8)
    for ap in APERTURES:
        f, e = s[f'flux_{ap}'], s[f'err_{ap}']
        v = valid_mask(f, e)
        rec[f'nvalid_{ap}'] = int(v.sum())
        rec[f'frac_zero_{ap}'] = float(np.mean(f == 0))
        rec[f'frac_neg_{ap}'] = float(np.mean(f[v] < 0)) if v.any() else np.nan
        rec[f'flux_med_{ap}'] = float(np.median(f[v])) if v.any() else np.nan
        rec[f'err_med_{ap}'] = float(np.median(e[v])) if v.any() else np.nan
        rec[f'sn_err_{ap}'] = float(np.median(f[v] / e[v])) if v.any() else np.nan
        rec[f'dersnr_{ap}'] = der_snr(f[v])
        if ap == '1RE' and v.any():
            idx = np.rint((w[v] - OBS_GRID[0]) / OBS_STEP).astype(int)
            obs_valid[idx[(idx >= 0) & (idx < len(OBS_GRID))]] = 1
        if np.isfinite(zf):
            wr = w / (1 + zf)
            rest[ap] = (to_grid(wr, f * (1 + zf), v, REST_GRID, 2.5 * step / (1 + zf)).astype(np.float32),
                        to_grid(wr, e * (1 + zf), v, REST_GRID, 2.5 * step / (1 + zf)).astype(np.float32))
    rec['ok'] = True
    return rec, rest, obs_valid

# ---- work list: galaxies with metadata AND spectrum
work = meta.merge(inv[['name', 'path']], on='name', how='inner')
if MAX_FILES: work = work.sample(min(MAX_FILES, len(work)), random_state=SEED)
work = work.sort_values('name').reset_index(drop=True)
N, P = len(work), len(REST_GRID)
tag = f'N{N}_P{P}_z{REST_GRID[0]:.0f}-{REST_GRID[-1]:.0f}'
cf = {'names': CACHE_DIR / f'names_{tag}.csv', 'qc': CACHE_DIR / f'qc_{tag}.parquet', 'obs': CACHE_DIR / f'obsvalid_{tag}.npy'}
for ap in APERTURES:
    cf[f'F_{ap}'] = CACHE_DIR / f'flux_{ap}_{tag}.npy'; cf[f'E_{ap}'] = CACHE_DIR / f'err_{ap}_{tag}.npy'

use_cache = (not FORCE_RELOAD) and all(p.exists() for p in cf.values()) \
            and pd.read_csv(cf['names'])['name'].tolist() == work['name'].tolist()
FLUX, ERR = {}, {}
if use_cache:
    qc = pd.read_parquet(cf['qc']); obs_valid_count = np.load(cf['obs'])
    for ap in APERTURES: FLUX[ap] = np.load(cf[f'F_{ap}']); ERR[ap] = np.load(cf[f'E_{ap}'])
    print(f'Loaded cache for {N} galaxies ({tag})')
else:
    for ap in APERTURES:
        FLUX[ap] = np.full((N, P), np.nan, np.float32); ERR[ap] = np.full((N, P), np.nan, np.float32)
    obs_valid_count = np.zeros(len(OBS_GRID), np.int64); recs = [None] * N
    t0 = time.time()
    with ThreadPoolExecutor(max_workers=N_THREADS) as ex:
        futs = {ex.submit(process_one, n, p, z): i
                for i, (n, p, z) in enumerate(zip(work['name'], work['path'], work['nsa_z']))}
        for fu in tqdm(as_completed(futs), total=N, desc='spectra'):
            i = futs[fu]; rec, rest, ov = fu.result(); recs[i] = rec
            if rest:
                for ap, (fr, er) in rest.items(): FLUX[ap][i] = fr; ERR[ap][i] = er
            if ov is not None: obs_valid_count += ov
    qc = pd.DataFrame(recs)
    print(f'Processed {N} spectra in {time.time()-t0:.0f}s')
    pd.DataFrame({'name': work['name']}).to_csv(cf['names'], index=False)
    qc.to_parquet(cf['qc']); np.save(cf['obs'], obs_valid_count)
    for ap in APERTURES: np.save(cf[f'F_{ap}'], FLUX[ap]); np.save(cf[f'E_{ap}'], ERR[ap])

df = work.drop(columns='path').merge(qc, on='name', how='left')
assert df['name'].tolist() == work['name'].tolist()
print('Failed reads:', int((~df['ok'].astype(bool)).sum()))
if 'error' in df: display(df.loc[df['error'].notna(), ['name', 'error']].head())

In [ ]:
# Rest-frame normalisation & S/N in the normalisation window
G = REST_GRID
nw = (G >= NORM_WINDOW[0]) & (G <= NORM_WINDOW[1])
NFLUX = {}
for ap in APERTURES:
    norm = np.nanmedian(FLUX[ap][:, nw], axis=1)
    norm[~(norm > 0)] = np.nan
    NFLUX[ap] = (FLUX[ap] / norm[:, None]).astype(np.float32)
    df[f'norm_{ap}'] = norm
    df[f'sn_rest_{ap}'] = np.nanmedian(FLUX[ap][:, nw] / ERR[ap][:, nw], axis=1)
    df[f'restcov_{ap}'] = np.isfinite(FLUX[ap]).mean(1)

df['good'] = df['ok'].astype(bool) & np.isfinite(df['norm_1RE']) & (df['dersnr_1RE'] >= SN_MIN)
df['good_FoV'] = df['ok'].astype(bool) & np.isfinite(df['norm_FoV']) & (df['dersnr_FoV'] >= SN_MIN)
print(f"good (1RE): {df['good'].sum()} / {len(df)} | good (FoV): {df['good_FoV'].sum()} / {len(df)}")
display(pd.crosstab(df['morph_class'], df['good'], margins=True))

## 5. Spectral data-quality EDA
Points to check:
* How many pixels are removed, and **where** in wavelength (instrumental vs. per-galaxy masks)?
* Is the S/N from the error column consistent with DER_SNR? In the sample files `ERROR_FoV` is several times larger than `FLUX_FoV` even though the spectra look smooth. A ratio well below 1 means the FoV errors are probably not 1σ per pixel (e.g. errors summed linearly over spaxels instead of in quadrature).
* Does data quality depend on redshift, mass or morphology? If it does, it can bias the ETG/LTG comparison.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes[0, 0].hist(df['n_pix'], bins=60, color='#5D6D7E'); axes[0, 0].set_title('pixels per spectrum')
axes[0, 1].hist(df['frac_removed_pix'] * 100, bins=60, color='#5D6D7E'); axes[0, 1].set_title('% pixels removed (gaps)')
axes[0, 2].hist(df['max_gap_A'], bins=60, color='#5D6D7E'); axes[0, 2].set_title('largest gap [Å]')
for ap, c in zip(APERTURES, ['k', 'purple']):
    axes[1, 0].hist(np.log10(df[f'sn_err_{ap}'].clip(1e-3)), bins=60, histtype='step', lw=2, color=c, label=f'{ap} (error col.)')
    axes[1, 0].hist(np.log10(df[f'dersnr_{ap}'].clip(1e-3)), bins=60, histtype='step', lw=2, ls='--', color=c, label=f'{ap} (DER_SNR)')
axes[1, 0].set_xlabel('log10 median S/N per pixel'); axes[1, 0].legend(fontsize=8); axes[1, 0].set_title('S/N distributions')
for ap, c in zip(APERTURES, ['k', 'purple']):
    axes[1, 1].scatter(df['dersnr_' + ap], df['sn_err_' + ap], s=3, alpha=.4, color=c, label=ap)
mx = np.nanmax(df[['dersnr_1RE', 'dersnr_FoV']].values)
axes[1, 1].plot([0.01, mx], [0.01, mx], 'r--', lw=1, label='1:1')
axes[1, 1].set_xscale('log'); axes[1, 1].set_yscale('log'); axes[1, 1].set_xlabel('DER_SNR'); axes[1, 1].set_ylabel('S/N from error column')
axes[1, 1].legend(); axes[1, 1].set_title('Are the error columns realistic?')
sns.boxplot(data=df, x='morph_group', y='dersnr_1RE', ax=axes[1, 2], color='#AED6F1', fliersize=1, order=sorted(df.morph_group.unique()))
axes[1, 2].set_yscale('log'); axes[1, 2].tick_params(axis='x', rotation=30); axes[1, 2].set_title('DER_SNR (1RE) by morphology')
plt.tight_layout(); savefig('qc_overview'); plt.show()

ratio = (df['sn_err_FoV'] / df['dersnr_FoV']).median(), (df['sn_err_1RE'] / df['dersnr_1RE']).median()
print(f'median [S/N(error column) / DER_SNR]:  FoV = {ratio[0]:.2f}   1RE = {ratio[1]:.2f}   (≈1 means realistic errors)')
display(df[['n_pix', 'frac_removed_pix', 'n_gaps', 'max_gap_A', 'sn_err_1RE', 'dersnr_1RE', 'sn_err_FoV', 'dersnr_FoV', 'frac_neg_1RE', 'frac_neg_FoV']].describe().T.round(3))

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(15, 7))
axes[0].plot(OBS_GRID, obs_valid_count / max(df['ok'].sum(), 1), lw=.7, color='k')
for sky in [5577.3, 6300.3, 5889.95, 5895.92]:
    axes[0].axvline(sky, color='orange', ls=':', lw=1)
axes[0].set_ylabel('fraction valid (1RE)'); axes[0].set_xlabel('observed wavelength [Å]')
axes[0].set_title('Where are pixels masked? (observed frame; orange = strong sky lines 5577, 5890/5896, 6300 Å)')
for k in ['ETG', 'LTG']:
    m = (df['morph_class'] == k).values
    if m.any(): axes[1].plot(G, np.isfinite(FLUX['1RE'][m]).mean(0), color=CLASS_COLORS[k], label=k)
axes[1].set_ylabel('fraction covered'); axes[1].set_xlabel('rest-frame wavelength [Å]'); axes[1].legend()
axes[1].set_title('Rest-frame coverage by class (needed to measure each spectral index)')
plt.tight_layout(); savefig('qc_masks'); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(14, 4.2))
sc = axes[0].scatter(df['nsa_z'], df['dersnr_1RE'], c=df['LogMass'], s=4, cmap='viridis', alpha=.6)
plt.colorbar(sc, ax=axes[0], label='log M*'); axes[0].set_yscale('log'); axes[0].set_xlabel('z'); axes[0].set_ylabel('DER_SNR 1RE')
axes[0].axhline(SN_MIN, color='r', ls='--'); axes[0].set_title('S/N vs redshift')
for k in ['ETG', 'LTG']:
    sns.kdeplot(df.loc[df.morph_class == k, 'frac_removed_pix'] * 100, ax=axes[1], color=CLASS_COLORS[k], label=k, lw=2, warn_singular=False)
axes[1].set_xlabel('% pixels removed'); axes[1].legend(); axes[1].set_title('Masking vs class')
plt.tight_layout(); plt.show()

## 6. Global spectral EDA
Rest-frame spectra are normalised by their median flux in the rest-frame window from 5300 to 5700 Å, so galaxies of different brightness can be compared. All analyses from here on use the **1RE** aperture and `good` spectra, unless stated otherwise.

In [ ]:
good = df['good'].values
gidx = np.where(good)[0]
NF = NFLUX['1RE']

# Random gallery
sel = rng.choice(gidx, size=min(8, len(gidx)), replace=False)
fig, ax = plt.subplots(figsize=(15, 8))
for k, i in enumerate(sel):
    r = df.iloc[i]
    ax.plot(G, NF[i] + 1.2 * k, lw=.6, color=CLASS_COLORS[r['morph_class']])
    ax.text(G[-1] + 30, 1.2 * k + 1, f"{r['name']}\n{r['Type']}  z={r['nsa_z']:.3f}", fontsize=7, va='center')
for lab, l0 in LINES.items():
    ax.axvline(l0, color='grey', ls=':', lw=.6); ax.text(l0, 1.2 * len(sel) + .4, lab, rotation=90, fontsize=7, ha='center')
ax.set_xlabel('rest-frame wavelength [Å]'); ax.set_ylabel('normalised flux + offset'); ax.set_xlim(G[0], G[-1] + 900)
ax.set_title('Random gallery of normalised rest-frame spectra (red = ETG, blue = LTG)')
plt.tight_layout(); savefig('gallery'); plt.show()

# Global median stack and scatter
fig, ax = plt.subplots(figsize=(15, 4))
p16, p50, p84 = np.nanpercentile(NF[gidx], [16, 50, 84], axis=0)
ax.fill_between(G, p16, p84, color='grey', alpha=.3, label='16–84%'); ax.plot(G, p50, color='k', lw=.8, label='median')
ax.set_ylim(0, np.nanmax(p84) * 1.1); ax.legend(); ax.set_xlabel('rest-frame wavelength [Å]')
ax.set_title(f'Median stack of {len(gidx)} good spectra'); plt.tight_layout(); plt.show()

### 6.1 PCA of the normalised spectra
Only pixels covered in at least 90% of good spectra are used. Remaining NaNs are filled with the median spectrum. PCA is unsupervised: if PC1 tracks T-type, the dominant spectral variation in the sample is the early/late sequence.

In [ ]:
from sklearn.decomposition import PCA

cov = np.isfinite(NF[gidx]).mean(0)
pix = cov >= 0.90
X = NF[gidx][:, pix].astype(np.float64)
med = np.nanmedian(X, axis=0)
X = np.where(np.isfinite(X), X, med)
X = np.clip(X, -1, 6)
pca = PCA(n_components=10, svd_solver='randomized', random_state=SEED).fit(X)
PCS = pca.transform(X)
for j in range(5): df.loc[df.index[gidx], f'PC{j+1}'] = PCS[:, j]
print(f'{pix.sum()} pixels used | explained variance:', np.round(pca.explained_variance_ratio_[:6], 3))

fig = plt.figure(figsize=(16, 8))
ax = fig.add_subplot(2, 3, 1)
ax.plot(np.arange(1, 11), np.cumsum(pca.explained_variance_ratio_), 'o-'); ax.set_xlabel('n components'); ax.set_ylabel('cumulative EVR')
ax.set_title('Explained variance')
ax = fig.add_subplot(2, 3, (2, 3))
for j in range(4): ax.plot(G[pix], pca.components_[j] + 0.1 * j, lw=.7, label=f'PC{j+1}')
ax.legend(); ax.set_title('Eigenspectra (offset)'); ax.set_xlabel('rest-frame λ [Å]')
ax = fig.add_subplot(2, 3, 4)
sc = ax.scatter(PCS[:, 0], PCS[:, 1], c=df['Ttype'].values[gidx], cmap=TTYPE_CMAP, vmin=-5, vmax=10, s=4, alpha=.7)
plt.colorbar(sc, ax=ax, label='T-type'); ax.set_xlabel('PC1'); ax.set_ylabel('PC2'); ax.set_title('PC1–PC2 coloured by T-type')
ax = fig.add_subplot(2, 3, 5)
sc = ax.scatter(PCS[:, 0], PCS[:, 1], c=df['LogMass'].values[gidx], cmap='viridis', s=4, alpha=.7)
plt.colorbar(sc, ax=ax, label='log M*'); ax.set_xlabel('PC1'); ax.set_title('PC1–PC2 coloured by mass')
ax = fig.add_subplot(2, 3, 6)
sc = ax.scatter(PCS[:, 0], PCS[:, 1], c=df['nsa_z'].values[gidx], cmap='magma', s=4, alpha=.7)
plt.colorbar(sc, ax=ax, label='z'); ax.set_xlabel('PC1'); ax.set_title('PC1–PC2 coloured by redshift')
plt.tight_layout(); savefig('pca'); plt.show()

## 7. Spectral feature measurement
All features are measured **vectorised over the whole sample** on the rest-frame grid. Band definitions are given in air wavelengths and converted to vacuum when `WAVE_IN_VACUUM=True`. A feature is set to NaN when any of its bands is less than 80% covered.

| Feature | Definition | Physical meaning |
|---|---|---|
| `D4000n` | ⟨F_ν⟩ 4000–4100 / ⟨F_ν⟩ 3850–3950 (Balogh+99) | age of the stellar population (high = old) |
| `HdA` | Lick Hδ_A (Worthey & Ottaviani 97) | A stars, i.e. star formation in the last ~1 Gyr |
| `Mgb`, `Fe5270`, `Fe5335`, `MgFe` | Lick indices, [MgFe]′ (Thomas+03) | metallicity / α-enhancement |
| `NaD` | Lick NaD | stars + interstellar medium (dust/gas) |
| `EW_Ha`, `EW_NII`, `EW_OII`, `EW_OIII` | pseudo-EW, **emission > 0** | ongoing star formation / nuclear activity |
| `color_BR` | −2.5 log(⟨F⟩4400–4600 / ⟨F⟩6000–6200) | spectral colour (higher = redder) |

> These are fast EDA estimates on the MaNGA spectral resolution, **not** calibrated to the Lick system and not corrected for stellar absorption under the emission lines. For publication-quality measurements, use the MaNGA DAP or a pPXF fit.

In [ ]:
def air_to_vac(lam):
    lam = np.asarray(lam, float); s2 = (1e4 / lam) ** 2
    n = 1 + 0.00008336624212083 + 0.02408926869968 / (130.1065924522 - s2) + 0.0001599740894897 / (38.92568793293 - s2)
    return lam * n
Wv = (lambda b: air_to_vac(b)) if WAVE_IN_VACUUM else (lambda b: np.asarray(b, float))
DL = float(np.median(np.diff(G))); MIN_COV = 0.8

def band_mean(F, band, p=0):
    a, b = Wv(band); c = (G >= a) & (G <= b)
    sub = F[:, c].astype(np.float64) * (G[c] ** p if p else 1)
    m = np.nanmean(sub, axis=1); m[np.isfinite(sub).mean(1) < MIN_COV] = np.nan
    return m

def pseudo_ew(F, blue, band, red, emission=False):
    cb, cr = band_mean(F, blue), band_mean(F, red)
    lb, lr = Wv(blue).mean(), Wv(red).mean()
    a, b = Wv(band); c = (G >= a) & (G <= b); lam = G[c]
    cont = cb[:, None] + (cr - cb)[:, None] * (lam[None, :] - lb) / (lr - lb)
    sub = F[:, c].astype(np.float64)
    ew = np.nanmean(1 - sub / cont, axis=1) * (b - a)
    ew[(np.isfinite(sub).mean(1) < MIN_COV) | ~np.isfinite(cb) | ~np.isfinite(cr)] = np.nan
    return -ew if emission else ew

LICK = {  # name: (blue, band, red) in air
    'HdA':    ((4041.60, 4079.75), (4083.50, 4122.25), (4128.50, 4161.00)),
    'HgA':    ((4283.50, 4319.75), (4319.75, 4363.50), (4367.25, 4419.75)),
    'Hb':     ((4827.875, 4847.875), (4847.875, 4876.625), (4876.625, 4891.625)),
    'Mgb':    ((5142.625, 5161.375), (5160.125, 5192.625), (5191.375, 5206.375)),
    'Fe5270': ((5233.150, 5248.150), (5245.650, 5285.650), (5285.650, 5318.150)),
    'Fe5335': ((5304.625, 5315.875), (5312.125, 5352.125), (5353.375, 5363.375)),
    'NaD':    ((5860.625, 5875.625), (5876.875, 5909.375), (5922.125, 5948.125)),
}
EMIS = {
    'EW_OII':  ((3696, 3716), (3716, 3738), (3738, 3758)),
    'EW_OIII': ((4978, 4993), (4997, 5017), (5020, 5040)),
    'EW_Ha':   ((6480, 6530), (6554, 6572), (6600, 6650)),
    'EW_NII':  ((6480, 6530), (6575, 6591), (6600, 6650)),
}

def compute_features(F):
    out = {'D4000n': band_mean(F, (4000, 4100), p=2) / band_mean(F, (3850, 3950), p=2)}
    for k, (bl, bd, rd) in LICK.items(): out[k] = pseudo_ew(F, bl, bd, rd)
    for k, (bl, bd, rd) in EMIS.items(): out[k] = pseudo_ew(F, bl, bd, rd, emission=True)
    out['color_BR'] = -2.5 * np.log10(band_mean(F, (4400, 4600)) / band_mean(F, (6000, 6200)))
    o = pd.DataFrame(out)
    fe = 0.72 * o['Fe5270'] + 0.28 * o['Fe5335']
    o['MgFe'] = np.sqrt((o['Mgb'] * fe).where((o['Mgb'] > 0) & (fe > 0)))
    o['log_NII_Ha'] = np.log10((o['EW_NII'] / o['EW_Ha']).where((o['EW_NII'] > 0) & (o['EW_Ha'] > 0)))
    o['log_EW_Ha'] = np.log10(o['EW_Ha'].where(o['EW_Ha'] > 0))
    return o

FEATS = ['D4000n', 'HdA', 'HgA', 'Hb', 'Mgb', 'Fe5270', 'Fe5335', 'MgFe', 'NaD',
         'EW_OII', 'EW_OIII', 'EW_Ha', 'EW_NII', 'log_NII_Ha', 'log_EW_Ha', 'color_BR']
feat = {ap: compute_features(FLUX[ap]) for ap in APERTURES}
for ap in APERTURES:
    f = feat[ap].copy(); f.columns = [f'{c}_{ap}' if ap != '1RE' else c for c in f.columns]
    df = pd.concat([df.drop(columns=[c for c in f.columns if c in df], errors='ignore'), f], axis=1)

# WHAN classes (Cid Fernandes+11) from the 1RE spectrum
def whan(x, ew, ew_nii):
    if not np.isfinite(ew): return 'n/a'
    if ew < 0.5 and (not np.isfinite(ew_nii) or ew_nii < 0.5): return 'passive'
    if ew < 3: return 'retired'
    if not np.isfinite(x): return 'n/a'
    if x < -0.4: return 'SF'
    return 'sAGN' if ew > 6 else 'wAGN'
df['WHAN'] = [whan(x, e, n) for x, e, n in zip(df['log_NII_Ha'], df['EW_Ha'], df['EW_NII'])]

display(df.loc[df.good, FEATS].describe(percentiles=[.05, .5, .95]).T.round(3))
print('Feature availability (% finite, good spectra):')
display((df.loc[df.good, FEATS].notna().mean() * 100).round(1).to_frame('% finite').T)

In [ ]:
fig, axes = plt.subplots(3, 6, figsize=(20, 9)); axes = axes.ravel()
for ax, c in zip(axes, FEATS + ['WHAN']):
    if c == 'WHAN':
        vc = df.loc[df.good, 'WHAN'].value_counts(); ax.bar(vc.index, vc.values, color='#5D6D7E'); ax.tick_params(axis='x', rotation=45)
    else:
        x = df.loc[df.good, c].dropna()
        if len(x): ax.hist(x.clip(*x.quantile([.01, .99])), bins=50, color='#5D6D7E')
    ax.set_title(c)
for ax in axes[len(FEATS) + 1:]: ax.axis('off')
plt.suptitle('Spectral feature distributions (1RE, good spectra)', fontweight='bold'); plt.tight_layout(); savefig('features_hist'); plt.show()

cc = df.loc[df.good, FEATS + ['Ttype', 'LogMass', 'nsa_z', 'incl', 'R90_kpc']].astype(float).corr(method='spearman')
plt.figure(figsize=(14, 11)); sns.heatmap(cc, cmap='vlag', center=0, annot=True, fmt='.2f', annot_kws={'size': 7})
plt.title('Spearman correlations: spectral features × metadata'); plt.tight_layout(); savefig('features_corr'); plt.show()

---
## 8. 🌗 How the data relate to Early-Type (ETG) vs Late-Type (LTG) classification

**Definitions (configurable in Section 0):** ETG = `Ttype ≤ 0` (E, S0, S0a); LTG = `1 ≤ Ttype ≤ 10` (Sa → Irr).

**Physical expectations to test against the data:**

| Observable | ETG (E/S0) | LTG (spirals) |
|---|---|---|
| Stellar mass, size | more massive, compact | broader range, lower mass |
| D4000n, colour | high (old, red) | lower (young, blue) |
| Hδ_A | low/negative | high (recent star formation) |
| Mgb, [MgFe]′ | strong (metal-rich, α-enhanced) | weaker |
| Hα, [OII] emission | weak (passive/retired) | strong (star-forming) |
| Inner (1RE) vs total (FoV) | flat | older bulge, younger disk (gradient) |
| Inclination | rounder | edge-on dust reddens spectra |

The subsections below check each row of this table. They then quantify how separable the classes are and point out the interesting exceptions.

### 8.1 Class definitions & balance

In [ ]:
CL = ['ETG', 'LTG']
d = df[df['morph_class'].isin(CL)].copy()
dg = d[d['good']].copy()
print(f"ETG/LTG with spectra: {len(d)} | with good spectra: {len(dg)}")
display(pd.crosstab(d['morph_group'], d['morph_class'], margins=True))

fig, axes = plt.subplots(1, 2, figsize=(15, 4))
vc = d.groupby(['Ttype', 'morph_class']).size().unstack(fill_value=0)
vc.plot.bar(stacked=True, ax=axes[0], color=[CLASS_COLORS[c] for c in vc.columns], edgecolor='k', width=.85)
axes[0].set_title('T-type counts by class'); axes[0].set_ylabel('N')
bar = d[d.morph_class == 'LTG']['bar'].value_counts(normalize=True)
bar_etg = d[d.morph_class == 'ETG']['bar'].value_counts(normalize=True)
pd.DataFrame({'ETG': bar_etg, 'LTG': bar}).fillna(0).T.plot.barh(stacked=True, ax=axes[1], colormap='Set2', edgecolor='k')
axes[1].set_title('Bar fraction from the Hubble Type string'); axes[1].set_xlabel('fraction'); axes[1].legend(fontsize=8, loc='lower right')
plt.tight_layout(); savefig('etg_ltg_balance'); plt.show()

### 8.2 Global (metadata) properties of ETGs vs LTGs
The table below gives medians, the Mann–Whitney p-value, the **AUC** (the probability that a random ETG has a higher value than a random LTG; 0.5 = no separation) and Cliff's δ = 2·AUC − 1.

In [ ]:
from sklearn.metrics import roc_auc_score

def compare(dd, cols):
    rows = []
    y = (dd['morph_class'] == 'ETG').astype(int).values
    for c in cols:
        x = dd[c].astype(float).values; m = np.isfinite(x)
        if m.sum() < 10 or len(np.unique(y[m])) < 2: continue
        a, b = x[m & (y == 1)], x[m & (y == 0)]
        auc = roc_auc_score(y[m], x[m])
        rows.append({'feature': c, 'N_ETG': len(a), 'N_LTG': len(b), 'median_ETG': np.median(a), 'median_LTG': np.median(b),
                     'MWU_p': stats.mannwhitneyu(a, b).pvalue, 'KS_D': stats.ks_2samp(a, b).statistic,
                     'AUC(ETG>LTG)': auc, 'cliffs_delta': 2 * auc - 1, 'separability': abs(2 * auc - 1)})
    return pd.DataFrame(rows).set_index('feature').sort_values('separability', ascending=False)

META_FEATS = ['LogMass', 'nsa_z', 'PETRO_TH90', 'R90_kpc', 'incl']
display(compare(d, META_FEATS).style.format('{:.3g}').background_gradient(subset=['separability'], cmap='Greens'))

fig, axes = plt.subplots(1, 5, figsize=(20, 3.8))
for ax, c in zip(axes, META_FEATS):
    for k in CL:
        x = d.loc[d.morph_class == k, c].dropna()
        if len(x) > 1: sns.kdeplot(x.clip(*d[c].quantile([.005, .995])), ax=ax, fill=True, alpha=.3, color=CLASS_COLORS[k], label=k, warn_singular=False)
    ax.set_title(c); ax.legend()
plt.tight_layout(); savefig('etg_ltg_meta'); plt.show()

### 8.3 Stacked spectra: ETG vs LTG, and along the Hubble sequence

In [ ]:
def stack(mask):
    return np.nanmedian(NF[mask], axis=0), np.nanpercentile(NF[mask], [25, 75], axis=0)

mE = (df.morph_class == 'ETG').values & good; mL = (df.morph_class == 'LTG').values & good
sE, qE = stack(mE); sL, qL = stack(mL)
fig = plt.figure(figsize=(16, 10)); gs = fig.add_gridspec(3, 3, height_ratios=[2.2, 1, 1.8])
ax = fig.add_subplot(gs[0, :])
for s, q, k, n in [(sE, qE, 'ETG', mE.sum()), (sL, qL, 'LTG', mL.sum())]:
    ax.fill_between(G, q[0], q[1], color=CLASS_COLORS[k], alpha=.2); ax.plot(G, s, color=CLASS_COLORS[k], lw=.8, label=f'{k} median (N={n})')
for lab, l0 in LINES.items(): ax.axvline(l0, color='grey', ls=':', lw=.5); ax.text(l0, 2.15, lab, rotation=90, fontsize=7, ha='center')
ax.set_ylim(0, 2.3); ax.legend(); ax.set_ylabel('normalised flux'); ax.set_title('Median stacked rest-frame spectra (shaded = IQR)')
ax = fig.add_subplot(gs[1, :], sharex=fig.axes[0])
ax.plot(G, sE / sL, color='k', lw=.7); ax.axhline(1, color='r', ls='--', lw=.8); ax.set_ylabel('ETG / LTG'); ax.set_xlabel('rest-frame λ [Å]')
for j, (lo, hi, t) in enumerate([(3680, 4200, 'D4000, CaII H&K, Hδ, [OII]'), (4800, 5300, 'Hβ, [OIII], Mgb, Fe'), (6450, 6780, 'Hα, [NII], [SII]')]):
    ax = fig.add_subplot(gs[2, j]); c = (G >= lo) & (G <= hi)
    ax.plot(G[c], sE[c], color=CLASS_COLORS['ETG'], label='ETG'); ax.plot(G[c], sL[c], color=CLASS_COLORS['LTG'], label='LTG')
    ax.set_title(t, fontsize=10); ax.set_xlabel('λ [Å]')
fig.axes[-3].legend(); plt.tight_layout(); savefig('etg_ltg_stacks'); plt.show()

# stacks per T-type
tts = sorted(df.loc[good & df.Ttype.notna().values, 'Ttype'].unique())
fig, axes = plt.subplots(1, 4, figsize=(20, 5), gridspec_kw={'width_ratios': [2.5, 1, 1, 1]})
norm_t = mpl.colors.Normalize(-5, 10)
for t in tts:
    m = good & (df.Ttype == t).values
    if m.sum() < 5: continue
    s = np.nanmedian(NF[m], axis=0)
    for ax, (lo, hi) in zip(axes, [(G[0], G[-1]), (3700, 4200), (4800, 5250), (6480, 6650)]):
        c = (G >= lo) & (G <= hi); ax.plot(G[c], s[c], color=TTYPE_CMAP(norm_t(t)), lw=.8)
sm = mpl.cm.ScalarMappable(norm=norm_t, cmap=TTYPE_CMAP); plt.colorbar(sm, ax=axes[-1], label='T-type')
axes[0].set_title('Median stack per T-type'); axes[1].set_title('4000 Å break'); axes[2].set_title('Hβ–[OIII]–Mgb'); axes[3].set_title('Hα + [NII]')
for ax in axes: ax.set_xlabel('rest λ [Å]')
plt.tight_layout(); savefig('stacks_ttype'); plt.show()

### 8.4 Spectral indices by class

In [ ]:
SPEC_FEATS = ['D4000n', 'HdA', 'Mgb', 'MgFe', 'NaD', 'color_BR', 'EW_Ha', 'EW_OII', 'EW_OIII', 'log_NII_Ha', 'HgA', 'Hb', 'Fe5270', 'Fe5335']
cmp_spec = compare(dg, SPEC_FEATS + META_FEATS + [c for c in ['PC1', 'PC2', 'PC3'] if c in dg])
display(cmp_spec.style.format('{:.3g}').background_gradient(subset=['separability'], cmap='Greens'))

show = ['D4000n', 'HdA', 'Mgb', 'MgFe', 'color_BR', 'EW_Ha', 'EW_OII', 'log_NII_Ha']
fig, axes = plt.subplots(1, len(show), figsize=(22, 4.5))
for ax, c in zip(axes, show):
    dd = dg[[c, 'morph_class']].dropna()
    if len(dd):
        lo, hi = dd[c].quantile([.01, .99]); dd[c] = dd[c].clip(lo, hi)
        sns.violinplot(data=dd, x='morph_class', y=c, order=CL, palette=CLASS_COLORS, ax=ax, cut=0, inner='quartile', hue='morph_class', legend=False)
    ax.set_xlabel(''); ax.set_title(f"{c}\nAUC={cmp_spec.loc[c, 'AUC(ETG>LTG)']:.2f}" if c in cmp_spec.index else c, fontsize=10)
plt.tight_layout(); savefig('etg_ltg_violins'); plt.show()

### 8.5 Diagnostic planes
* **D4000n vs Hδ_A** (Kauffmann+03) traces star-formation history. ETGs should sit in the old/quiescent corner (high D4000, low Hδ_A).
* **D4000n vs stellar mass** shows the red-sequence/blue-cloud bimodality. Morphology and mass are entangled.
* **WHAN diagram** (Cid Fernandes+11): log [NII]/Hα vs log EW(Hα) separates star-forming, AGN, retired and passive galaxies.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
for k in ['LTG', 'ETG']:
    s = dg[dg.morph_class == k]
    axes[0].scatter(s['D4000n'], s['HdA'], s=4, alpha=.35, color=CLASS_COLORS[k], label=k)
    axes[1].scatter(s['LogMass'], s['D4000n'], s=4, alpha=.35, color=CLASS_COLORS[k], label=k)
    axes[2].scatter(s['log_NII_Ha'], s['log_EW_Ha'], s=4, alpha=.35, color=CLASS_COLORS[k], label=k)
for k, ls in zip(CL, ['-', '--']):
    s = dg[dg.morph_class == k][['D4000n', 'HdA']].dropna()
    if len(s) > 20:
        sns.kdeplot(data=s, x='D4000n', y='HdA', ax=axes[0], levels=[.2, .5, .8], color=CLASS_COLORS[k], linestyles=ls, warn_singular=False)
axes[0].set_xlim(*dg['D4000n'].quantile([.005, .995])); axes[0].set_ylim(*dg['HdA'].quantile([.005, .995]))
axes[0].set_xlabel('D4000n'); axes[0].set_ylabel('Hδ_A [Å]'); axes[0].set_title('Star-formation history plane'); axes[0].legend(markerscale=4)
axes[1].set_ylim(*dg['D4000n'].quantile([.005, .995])); axes[1].set_xlabel('log M*'); axes[1].set_ylabel('D4000n'); axes[1].set_title('Age–mass bimodality')
xx = np.linspace(-1.5, 1, 10)
axes[2].axhline(np.log10(3), color='k', lw=1); axes[2].plot([-0.4, -0.4], [np.log10(3), 3], 'k', lw=1); axes[2].plot([-0.4, 1], [np.log10(6)] * 2, 'k', lw=1)
for t, (x, y) in {'SF': (-1.2, 2.2), 'sAGN': (0.3, 2.2), 'wAGN': (0.3, 0.62), 'retired/passive': (-1.2, 0.1)}.items():
    axes[2].text(x, y, t, fontsize=11, fontweight='bold')
axes[2].set_xlim(-1.5, 1); axes[2].set_ylim(-1, 2.6); axes[2].set_xlabel('log [NII]/Hα (EW ratio)'); axes[2].set_ylabel('log EW(Hα) [Å]'); axes[2].set_title('WHAN diagram')
plt.tight_layout(); savefig('etg_ltg_planes'); plt.show()

wt = pd.crosstab(dg['morph_class'], dg['WHAN'], normalize='index').mul(100).round(1)
print('WHAN class (%) per morphology:'); display(wt)

### 8.6 T-type as a continuous sequence
Medians and interquartile ranges of each feature per T-type, with Spearman ρ. A monotonic trend means the spectra encode the position of a galaxy along the Hubble sequence, not just an ETG/LTG dichotomy.

In [ ]:
seq = ['D4000n', 'HdA', 'MgFe', 'color_BR', 'log_EW_Ha', 'EW_OII', 'LogMass', 'R90_kpc']
dt = df[good & df.morph_class.isin(CL).values]
fig, axes = plt.subplots(2, 4, figsize=(20, 8)); axes = axes.ravel()
rho = {}
for ax, c in zip(axes, seq):
    g = dt.groupby('Ttype')[c]; q = g.quantile([.25, .5, .75]).unstack()
    ax.errorbar(q.index, q[0.5], yerr=[q[0.5] - q[0.25], q[0.75] - q[0.5]], fmt='o-', color='k', capsize=3)
    ax.scatter(q.index, q[0.5], c=[TTYPE_CMAP(norm_t(t)) for t in q.index], s=80, zorder=3, edgecolor='k')
    ax.axvspan(-6, ETG_MAX_TTYPE + .5, color=CLASS_COLORS['ETG'], alpha=.07)
    m = dt[[c, 'Ttype']].dropna(); rho[c] = stats.spearmanr(m['Ttype'], m[c])[0] if len(m) > 3 else np.nan
    ax.set_title(f'{c}   ρ={rho[c]:+.2f}'); ax.set_xlabel('T-type')
plt.tight_layout(); savefig('ttype_sequence'); plt.show()

### 8.7 Controlling for stellar mass
ETGs are on average more massive, so part of their spectral difference could be a **mass effect**. Here we compare classes **at fixed mass**. If the gap between classes remains in every mass bin, morphology carries information beyond mass.

In [ ]:
bins = np.arange(np.floor(dg.LogMass.min() * 4) / 4, dg.LogMass.max() + .25, .25)
dg['mbin'] = pd.cut(dg['LogMass'], bins)
fig, axes = plt.subplots(1, 3, figsize=(19, 4.8))
for ax, c in zip(axes, ['D4000n', 'log_EW_Ha', 'HdA']):
    for k in CL:
        s = dg[dg.morph_class == k].groupby('mbin', observed=True)[c]
        q = s.quantile([.25, .5, .75]).unstack(); n = s.count()
        q = q[n.reindex(q.index) >= 10]
        x = [i.mid for i in q.index]
        ax.plot(x, q[0.5], 'o-', color=CLASS_COLORS[k], label=k); ax.fill_between(x, q[0.25], q[0.75], color=CLASS_COLORS[k], alpha=.2)
    ax.set_xlabel('log M*'); ax.set_title(f'{c} at fixed mass'); ax.legend()
plt.tight_layout(); savefig('mass_controlled'); plt.show()

# mass-matched comparison: for every ETG pick the nearest-mass LTG (without replacement)
from sklearn.neighbors import NearestNeighbors
e = dg[dg.morph_class == 'ETG'].dropna(subset=['LogMass']); l = dg[dg.morph_class == 'LTG'].dropna(subset=['LogMass'])
if len(e) and len(l):
    nn = NearestNeighbors(n_neighbors=min(10, len(l))).fit(l[['LogMass']])
    dist, ind = nn.kneighbors(e[['LogMass']]); used, pick = set(), []
    for row in ind:
        j = next((j for j in row if j not in used), None)
        if j is not None: used.add(j); pick.append(j)
    matched = pd.concat([e.iloc[:len(pick)], l.iloc[pick]])
    print(f'Mass-matched sample: {len(pick)} ETG + {len(pick)} LTG')
    display(compare(matched, ['D4000n', 'HdA', 'MgFe', 'color_BR', 'log_EW_Ha', 'EW_OII', 'LogMass'])[['median_ETG', 'median_LTG', 'AUC(ETG>LTG)', 'separability']].round(3))

### 8.8 Inner (1 Rₑ) vs full field-of-view spectra
This uses only galaxies whose FoV spectrum passes the S/N cut. ΔD4000 = D4000(1RE) − D4000(FoV). A positive value means the centre is older than the galaxy as a whole, which is the bulge–disk (inside-out) signature expected in LTGs.

In [ ]:
da = df[df.good & df.good_FoV & df.morph_class.isin(CL)].copy()
da['dD4000'] = da['D4000n'] - da['D4000n_FoV']
da['dEW_Ha'] = da['log_EW_Ha'] - da['log_EW_Ha_FoV']
da['dColor'] = da['color_BR'] - da['color_BR_FoV']
print(f'{len(da)} galaxies with good 1RE and FoV spectra')
if len(da) > 10:
    display(compare(da, ['dD4000', 'dEW_Ha', 'dColor', 'D4000n_FoV', 'log_EW_Ha_FoV'])[['median_ETG', 'median_LTG', 'MWU_p', 'AUC(ETG>LTG)']].round(3))
    fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
    for k in CL:
        s = da[da.morph_class == k]
        axes[0].scatter(s['D4000n_FoV'], s['D4000n'], s=4, alpha=.4, color=CLASS_COLORS[k], label=k)
        for ax, c in zip(axes[1:], ['dD4000', 'dEW_Ha']):
            x = s[c].dropna()
            if len(x) > 1: sns.kdeplot(x.clip(*da[c].quantile([.01, .99])), ax=ax, color=CLASS_COLORS[k], fill=True, alpha=.3, label=k, warn_singular=False)
    lim = da[['D4000n', 'D4000n_FoV']].quantile([.01, .99]).values
    axes[0].plot([lim.min(), lim.max()], [lim.min(), lim.max()], 'k--'); axes[0].set_xlabel('D4000n (FoV)'); axes[0].set_ylabel('D4000n (1RE)'); axes[0].legend(markerscale=4)
    axes[1].axvline(0, color='k', lw=.8); axes[1].set_title('ΔD4000 = inner − total'); axes[1].legend()
    axes[2].axvline(0, color='k', lw=.8); axes[2].set_title('Δlog EW(Hα) = inner − total'); axes[2].legend()
    plt.tight_layout(); savefig('aperture_gradients'); plt.show()

### 8.9 Inclination effects
Edge-on disks are reddened by dust, which pushes D4000 and colour towards ETG-like values. This is a known source of LTG→ETG confusion in spectral classification.

In [ ]:
dg['incl_bin'] = pd.cut(dg['incl'], [0, 30, 45, 60, 75, 90])
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
for ax, c in zip(axes, ['color_BR', 'D4000n', 'log_EW_Ha']):
    sns.boxplot(data=dg, x='incl_bin', y=c, hue='morph_class', hue_order=CL, palette=CLASS_COLORS, ax=ax, fliersize=0)
    ax.set_ylim(*dg[c].quantile([.02, .98])); ax.set_xlabel('inclination [deg]'); ax.set_title(f'{c} vs inclination')
plt.tight_layout(); savefig('inclination'); plt.show()

### 8.10 Unsupervised view: do the spectra form two groups by themselves?
A 2-component Gaussian Mixture is fitted on the first PCA components **without labels** and then compared with the visual ETG/LTG classes.

In [ ]:
from sklearn.mixture import GaussianMixture
from sklearn.metrics import adjusted_rand_score
pcs = [c for c in ['PC1', 'PC2', 'PC3', 'PC4', 'PC5'] if c in dg]
u = dg.dropna(subset=pcs)
if len(u) > 20:
    gm = GaussianMixture(2, random_state=SEED, n_init=5).fit(u[pcs])
    lab = gm.predict(u[pcs])
    # name clusters by their ETG fraction
    etg_frac = pd.Series(u['morph_class'].eq('ETG').values).groupby(lab).mean()
    names = {etg_frac.idxmax(): 'spec-early', etg_frac.idxmin(): 'spec-late'}
    u = u.assign(spec_cluster=[names[i] for i in lab])
    print('Adjusted Rand Index (clusters vs ETG/LTG):', round(adjusted_rand_score(u['morph_class'], u['spec_cluster']), 3))
    display(pd.crosstab(u['morph_class'], u['spec_cluster'], normalize='index').round(3))
    fig, axes = plt.subplots(1, 3, figsize=(18, 4.8))
    for k in CL:
        s = u[u.morph_class == k]; axes[0].scatter(s.PC1, s.PC2, s=4, alpha=.4, color=CLASS_COLORS[k], label=k)
        sns.kdeplot(s.PC1, ax=axes[1], color=CLASS_COLORS[k], fill=True, alpha=.3, label=k, warn_singular=False)
    axes[0].set_xlabel('PC1'); axes[0].set_ylabel('PC2'); axes[0].legend(markerscale=4); axes[0].set_title('PCA plane by visual class')
    axes[1].set_title('PC1 distribution'); axes[1].legend()
    sns.boxplot(data=u, x='Ttype', y='PC1', ax=axes[2], color='#AED6F1', fliersize=1)
    axes[2].set_title(f"PC1 vs T-type (ρ={stats.spearmanr(u.Ttype, u.PC1)[0]:+.2f})")
    plt.tight_layout(); savefig('unsupervised'); plt.show()

### 8.11 Supervised separability: how well do spectra predict ETG vs LTG?
We use a gradient-boosting classifier, which handles NaNs natively, with **plate-grouped** stratified 5-fold CV so that galaxies observed on the same plate never appear in both training and test folds. Three feature sets are compared:
1. **metadata only** (mass, z, size, inclination),
2. **spectral features only** (indices + PCs),
3. **both**.

Permutation importance shows which observables carry the morphological information. A regressor on T-type checks how much of the full Hubble sequence can be recovered from the spectra.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.model_selection import StratifiedGroupKFold, GroupKFold, cross_val_predict
from sklearn.metrics import roc_curve, balanced_accuracy_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.inspection import permutation_importance

SPEC_SET = ['D4000n', 'HdA', 'HgA', 'Hb', 'Mgb', 'Fe5270', 'Fe5335', 'NaD', 'EW_OII', 'EW_OIII', 'EW_Ha', 'EW_NII', 'color_BR'] + pcs
META_SET = ['LogMass', 'nsa_z', 'R90_kpc', 'PETRO_TH90', 'incl']
SETS = {'metadata only': META_SET, 'spectra only': SPEC_SET, 'spectra + metadata': SPEC_SET + META_SET}

ml = dg.copy(); y = (ml['morph_class'] == 'ETG').astype(int).values; groups = ml['plate'].astype(int).values
n_splits = int(min(5, np.bincount(y).min(), len(np.unique(groups))))
cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
res, probas = [], {}
fig, axes = plt.subplots(1, 3, figsize=(19, 5))
for name, cols in SETS.items():
    clf = HistGradientBoostingClassifier(max_iter=300, learning_rate=.05, class_weight='balanced', random_state=SEED)
    p = cross_val_predict(clf, ml[cols].astype(float), y, groups=groups, cv=cv, method='predict_proba')[:, 1]
    probas[name] = p
    res.append({'feature set': name, 'ROC AUC': roc_auc_score(y, p), 'balanced acc.': balanced_accuracy_score(y, p > .5)})
    fpr, tpr, _ = roc_curve(y, p); axes[0].plot(fpr, tpr, label=f"{name} (AUC={res[-1]['ROC AUC']:.3f})")
axes[0].plot([0, 1], [0, 1], 'k:'); axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR'); axes[0].set_title(f'ROC, {n_splits}-fold plate-grouped CV'); axes[0].legend(fontsize=8)
display(pd.DataFrame(res).set_index('feature set').round(3))

best = 'spectra + metadata'
ConfusionMatrixDisplay(confusion_matrix(y, probas[best] > .5, normalize='true'), display_labels=['LTG', 'ETG']).plot(ax=axes[1], cmap='Blues', colorbar=False)
axes[1].set_title(f'Confusion matrix ({best})')

# permutation importance on a single plate-grouped split
tr, te = next(StratifiedGroupKFold(n_splits=max(n_splits, 2), shuffle=True, random_state=SEED).split(ml, y, groups))
clf = HistGradientBoostingClassifier(max_iter=300, learning_rate=.05, class_weight='balanced', random_state=SEED)
cols = SETS[best]; clf.fit(ml.iloc[tr][cols].astype(float), y[tr])
pi = permutation_importance(clf, ml.iloc[te][cols].astype(float), y[te], scoring='roc_auc', n_repeats=10, random_state=SEED)
imp = pd.Series(pi.importances_mean, index=cols).sort_values()
imp.tail(15).plot.barh(ax=axes[2], color=['#C0392B' if c in META_SET else '#2E86C1' for c in imp.tail(15).index])
axes[2].set_title('Permutation importance (ΔAUC)\nred = metadata, blue = spectral')
plt.tight_layout(); savefig('classifier'); plt.show()
dg['p_ETG'] = probas[best]

# T-type regression from spectra only
reg = HistGradientBoostingRegressor(max_iter=300, learning_rate=.05, random_state=SEED)
tt_pred = cross_val_predict(reg, ml[SPEC_SET].astype(float), ml['Ttype'].values, groups=groups, cv=GroupKFold(n_splits=n_splits))
fig, ax = plt.subplots(figsize=(8, 4.5))
sns.boxplot(x=ml['Ttype'].values, y=tt_pred, ax=ax, color='#AED6F1', fliersize=1)
ax.set_xlabel('visual T-type'); ax.set_ylabel('T-type predicted from spectra')
ax.set_title(f"Spearman ρ = {stats.spearmanr(ml['Ttype'], tt_pred)[0]:.2f} | MAE = {np.mean(np.abs(ml['Ttype'] - tt_pred)):.2f}")
plt.tight_layout(); plt.show()

### 8.12 Interesting exceptions
* **Star-forming ETGs:** visually early-type, but with an SF-like WHAN class (EW(Hα) > 3 Å and log [NII]/Hα < −0.4). These are candidates for rejuvenation or gas accretion.
* **Passive LTGs** ("red spirals"): visually late-type with D4000n > 1.6 and EW(Hα) < 3 Å. Check their inclination, since dust in edge-on disks can mimic this.
* **Classifier disagreements:** objects the model is confident about but that carry the opposite visual label. These may be misclassifications or genuine transition objects.

In [ ]:
sf_etg = dg[(dg.morph_class == 'ETG') & (dg.WHAN == 'SF')]
red_ltg = dg[(dg.morph_class == 'LTG') & (dg.D4000n > 1.6) & (dg.EW_Ha < 3)]
mis = dg[((dg.morph_class == 'ETG') & (dg.p_ETG < .1)) | ((dg.morph_class == 'LTG') & (dg.p_ETG > .9))]
nE, nL = (dg.morph_class == 'ETG').sum(), (dg.morph_class == 'LTG').sum()
print(f'Star-forming ETGs : {len(sf_etg)} ({len(sf_etg)/max(nE,1):.1%} of ETGs)')
print(f'Passive LTGs      : {len(red_ltg)} ({len(red_ltg)/max(nL,1):.1%} of LTGs) | median incl = {red_ltg.incl.median():.0f}° vs {dg[dg.morph_class=="LTG"].incl.median():.0f}° for all LTGs')
print(f'Confident disagreements: {len(mis)}')
cols_show = ['name', 'Type', 'Ttype', 'nsa_z', 'LogMass', 'incl', 'D4000n', 'HdA', 'EW_Ha', 'WHAN', 'p_ETG']
display(mis.sort_values('p_ETG')[cols_show].head(15).round(3))
for nm, sub in [('sf_ETG', sf_etg), ('passive_LTG', red_ltg), ('classifier_disagreements', mis)]:
    sub[cols_show].to_csv(CACHE_DIR / f'{nm}.csv', index=False)

fig, axes = plt.subplots(2, 1, figsize=(15, 7), sharex=True)
for ax, sub, ttl in [(axes[0], sf_etg, 'Star-forming ETGs'), (axes[1], red_ltg, 'Passive LTGs')]:
    for k, i in enumerate(sub.index[:5]):
        ax.plot(G, NF[i] + .8 * k, lw=.6, color=CLASS_COLORS[df.loc[i, 'morph_class']])
        ax.text(G[-1] + 20, .8 * k + 1, f"{df.loc[i, 'name']} {df.loc[i, 'Type']}", fontsize=7)
    ax.plot(G, sE - .9, color='k', lw=.6, ls='--', label='ETG median'); ax.plot(G, sL - .9, color='grey', lw=.6, label='LTG median')
    ax.set_title(ttl); ax.legend(loc='upper right', fontsize=8)
axes[1].set_xlabel('rest-frame λ [Å]'); axes[1].set_xlim(G[0], G[-1] + 700)
plt.tight_layout(); savefig('exceptions'); plt.show()

### 8.13 Automatic summary of the ETG/LTG findings

In [ ]:
top = cmp_spec.head(6)
lines = [f"* Sample: **{len(dg)}** ETG/LTG galaxies with good 1RE spectra ({(dg.morph_class=='ETG').mean():.0%} ETG)."]
lines.append('* Most discriminating single observables (|Cliff δ|): ' +
             ', '.join(f"**{i}** ({r['separability']:.2f}; ETG median {r['median_ETG']:.2f} vs LTG {r['median_LTG']:.2f})" for i, r in top.iterrows()) + '.')
r = pd.DataFrame(res).set_index('feature set')['ROC AUC']
lines.append(f"* Classifier ROC AUC: metadata {r['metadata only']:.3f} | spectra {r['spectra only']:.3f} | both {r['spectra + metadata']:.3f}.")
lines.append('* Spearman ρ with T-type: ' + ', '.join(f'{k} {v:+.2f}' for k, v in rho.items() if np.isfinite(v)) + '.')
if len(wt): lines.append('* WHAN: ' + '; '.join(f"{k}: " + ', '.join(f'{c} {v:.0f}%' for c, v in wt.loc[k].items() if v >= 5) for k in wt.index) + '.')
lines.append(f"* Exceptions: {len(sf_etg)} star-forming ETGs, {len(red_ltg)} passive LTGs, {len(mis)} confident classifier disagreements (lists saved in `{CACHE_DIR}`).")
display(Markdown('\n'.join(lines)))

## 9. Export

In [ ]:
out_cols = [c for c in df.columns if c not in ('path',)]
df[out_cols].to_parquet(CACHE_DIR / 'galaxy_features_qc.parquet', index=False)
df[out_cols].to_csv(CACHE_DIR / 'galaxy_features_qc.csv', index=False)
np.save(CACHE_DIR / 'rest_grid.npy', REST_GRID)
print('Saved to', CACHE_DIR); print(sorted(p.name for p in CACHE_DIR.iterdir())[:30])